In [1]:
# Helper utilities for Colab (self-contained)
import math
import numpy as np
from scipy import sparse

def load_sparse_matrix(path: str) -> sparse.csr_matrix:
    matrix = sparse.load_npz(path)
    if not sparse.isspmatrix_csr(matrix):
        matrix = matrix.tocsr()
    return matrix

def _top_k_indices(scores: np.ndarray, k: int) -> np.ndarray:
    if k <= 0:
        return np.array([], dtype=np.int64)
    if k >= scores.shape[0]:
        return np.argsort(-scores)
    idx = np.argpartition(-scores, k - 1)[:k]
    return idx[np.argsort(-scores[idx])]

def compute_user_metrics(
    scores: np.ndarray,
    train_items: np.ndarray,
    test_items: np.ndarray,
    k: int,
    use_ndcg: bool = True,
) -> tuple[float, float, float] | None:
    if test_items.size == 0 or k <= 0 or scores.size == 0:
        return None
    scores = np.asarray(scores, dtype=float)
    scores = scores.copy()
    scores[train_items] = -np.inf
    k = min(k, scores.size)
    ranked = _top_k_indices(scores, k)
    test_set = set(test_items.tolist())
    hits = sum(1 for item in ranked if item in test_set)
    precision = hits / k
    recall = hits / len(test_set)
    ndcg = 0.0
    if use_ndcg:
        dcg = 0.0
        for rank, item in enumerate(ranked):
            if item in test_set:
                dcg += 1.0 / math.log2(rank + 2)
        ideal = min(len(test_set), k)
        idcg = sum(1.0 / math.log2(rank + 2) for rank in range(ideal))
        ndcg = (dcg / idcg) if idcg > 0 else 0.0
    return precision, recall, ndcg

def evaluate_ranking(
    score_fn,
    train_interactions: sparse.csr_matrix,
    test_interactions: sparse.csr_matrix,
    k: int,
    use_ndcg: bool = True,
) -> dict:
    precisions = []
    recalls = []
    ndcgs = []
    num_users = train_interactions.shape[0]
    for user_id in range(num_users):
        test_items = test_interactions[user_id].indices
        if test_items.size == 0:
            continue
        train_items = train_interactions[user_id].indices
        scores = score_fn(user_id)
        metrics = compute_user_metrics(scores, train_items, test_items, k, use_ndcg)
        if metrics is None:
            continue
        precision, recall, ndcg = metrics
        precisions.append(precision)
        recalls.append(recall)
        if use_ndcg:
            ndcgs.append(ndcg)
    result = {
        "k": float(k),
        "users_evaluated": float(len(precisions)),
        "precision_at_k": float(np.mean(precisions)) if precisions else 0.0,
        "recall_at_k": float(np.mean(recalls)) if recalls else 0.0,
    }
    if use_ndcg:
        result["ndcg_at_k"] = float(np.mean(ndcgs)) if ndcgs else 0.0
    return result

def recommend_top_k(scores: np.ndarray, train_items: np.ndarray, k: int) -> np.ndarray:
    scores = np.asarray(scores, dtype=float)
    scores = scores.copy()
    scores[train_items] = -np.inf
    k = min(k, scores.size)
    return _top_k_indices(scores, k)

In [2]:
# ==========================================
# 1. CÀI ĐẶT THƯ VIỆN & SETUP
# ==========================================
!pip install -q lightfm-next numpy scipy scikit-learn

import os
import numpy as np
import pickle
import time
from scipy import sparse
from lightfm import LightFM

# Cấu hình đường dẫn (Điều chỉnh nếu bạn dùng Google Drive)
DATA_DIR_P5 = "/content"
DATA_DIR_P4 = "/content"

# ==========================================
# 2. LOAD DỮ LIỆU
# ==========================================
print("--- Đang tải dữ liệu ---")
start_time = time.time()

try:
    train_interactions_csr = load_sparse_matrix(os.path.join(DATA_DIR_P5, "train_interactions.npz"))
    train_weights_csr = load_sparse_matrix(os.path.join(DATA_DIR_P5, "train_weights (1).npz"))
    test_interactions = load_sparse_matrix(os.path.join(DATA_DIR_P5, "test_interactions.npz"))

    # Item features dùng CSR để dự đoán nhanh
    item_features = sparse.load_npz(os.path.join(DATA_DIR_P4, "item_features (1).npz")).tocsr()
    if item_features.shape[0] != train_interactions_csr.shape[1]:
        raise ValueError(
            f"Item features rows ({item_features.shape[0]}) do not match items "
            f"({train_interactions_csr.shape[1]}). Check book2id alignment."
        )
    # Add identity features so each item has a unique embedding signal.
    ADD_IDENTITY_FEATURES = True
    if ADD_IDENTITY_FEATURES:
        identity = sparse.identity(item_features.shape[0], format="csr")
        item_features = sparse.hstack([item_features, identity], format="csr")
        print(f"✅ Added identity features: {item_features.shape}")

    # LightFM training expects COO for interactions/weights.
    train_interactions = train_interactions_csr.tocoo()
    train_weights = train_weights_csr.tocoo()

    print(f"✅ Đã load xong dữ liệu ({time.time() - start_time:.2f}s)")
    print(f"📊 Shape Interactions: {train_interactions_csr.shape}")
    print(f"📊 Shape Item Features: {item_features.shape}")
    print(f"📊 Số lượng tương tác: {train_interactions_csr.nnz}")
except FileNotFoundError as e:
    print(f"❌ Lỗi: Không tìm thấy file. Hãy kiểm tra đường dẫn folder {DATA_DIR_P4} và {DATA_DIR_P5}")
    raise e

# ==========================================
# 3. HUẤN LUYỆN MÔ HÌNH (Aligned với train_hybrid.py)
# ==========================================
print("\n--- Đang khởi tạo và huấn luyện mô hình ---")
np.random.seed(42)

model = LightFM(
    loss="warp",                # Thuật toán tối ưu cho bài toán Ranking
    no_components=64,           # Kích thước vector nhúng (latent factors)
    learning_rate=0.05,
    random_state=42,
 )

# Bắt đầu Fit
train_start = time.time()
model.fit(
    train_interactions,
    sample_weight=train_weights, # Sử dụng trọng số rating thực tế
    item_features=item_features,
    epochs=20,                   # Số vòng lặp huấn luyện (aligned)
    num_threads=4,
 )
print(f"✅ Huấn luyện hoàn tất! Thời gian: {(time.time() - train_start)/60:.2f} phút")

# ==========================================
# 4. ĐÁNH GIÁ (EVALUATION) - aligned với script
# ==========================================
print("\n--- Đang đánh giá hiệu năng (K=10) ---")
eval_start = time.time()

num_items = train_interactions_csr.shape[1]
item_ids = np.arange(num_items, dtype=np.int64)

def score_fn(user_id: int) -> np.ndarray:
    return model.predict(user_id, item_ids, item_features=item_features)

metrics = evaluate_ranking(
    score_fn,
    train_interactions_csr,
    test_interactions,
    k=10,
    use_ndcg=True,
 )

print("-" * 40)
print(f"🚀 Precision@10: {metrics['precision_at_k']:.6f}")
print(f"🚀 Recall@10:    {metrics['recall_at_k']:.6f}")
print(f"🚀 NDCG@10:      {metrics['ndcg_at_k']:.6f}")
print(f"⏱️ Thời gian đánh giá:   {time.time() - eval_start:.2f}s")
print("-" * 40)

# ==========================================
# 5. LƯU MÔ HÌNH (EXPORT)
# ==========================================
model_filename = "lightfm_model_final.pkl"
with open(model_filename, "wb") as f:
    pickle.dump(model, f)

print(f"\n✅ Đã lưu mô hình thành công vào file: {model_filename}")
print("Bạn có thể tải file này về để sử dụng cho Inference sau này.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 12.6 MB/s eta 0:00:00
--- Đang tải dữ liệu ---
✅ Added identity features: (20700, 21947)
✅ Đã load xong dữ liệu (0.18s)
📊 Shape Interactions: (79153, 20700)
📊 Shape Item Features: (20700, 21947)
📊 Số lượng tương tác: 1539303

--- Đang khởi tạo và huấn luyện mô hình ---
✅ Huấn luyện hoàn tất! Thời gian: 1.89 phút

--- Đang đánh giá hiệu năng (K=10) ---
----------------------------------------
🚀 Precision@10: 0.084923
🚀 Recall@10:    0.227997
🚀 NDCG@10:      0.188892
⏱️ Thời gian đánh giá:   276.67s
----------------------------------------

✅ Đã lưu mô hình thành công vào file: lightfm_model_final.pkl
Bạn có thể tải file này về để sử dụng cho Inference sau này.


In [3]:
# Config summary (for reproducibility)
config_summary = {
    "data_dir_p5": DATA_DIR_P5,
    "data_dir_p4": DATA_DIR_P4,
    "loss": "warp",
    "no_components": 64,
    "learning_rate": 0.05,
    "epochs": 20,
    "random_state": 42,
    "num_threads": 4,
    "k_eval": 10,
    "use_ndcg": True,
}

print("--- CONFIG SUMMARY ---")
for key, value in config_summary.items():
    print(f"{key}: {value}")

print("Train shape:", train_interactions_csr.shape)
print("Test shape:", test_interactions.shape)
print("Item features shape:", item_features.shape)

--- CONFIG SUMMARY ---
data_dir_p5: /content
data_dir_p4: /content
loss: warp
no_components: 64
learning_rate: 0.05
epochs: 20
random_state: 42
num_threads: 4
k_eval: 10
use_ndcg: True
Train shape: (79153, 20700)
Test shape: (79153, 20700)
Item features shape: (20700, 21947)


In [4]:
import os
import csv
import json
import numpy as np
from scipy import sparse
from lightfm import LightFM

# ====== Toggle diagnostics (optional) ======
K = 10
RUN_POP_BASELINE = False
RUN_STRONG_EVAL = False
RUN_ABLATION = False
RUN_LOSS_COMPARE = False

# ====== Sanity checks ======
print("\n--- SANITY CHECKS ---")
print("Train shape:", train_interactions_csr.shape)
print("Train nnz:", train_interactions_csr.nnz)
print("Test nnz:", test_interactions.nnz)
print("Item features shape:", item_features.shape)

train_test_overlap = train_interactions_csr.multiply(test_interactions).nnz
print("Train/Test overlap nnz:", train_test_overlap)

def eval_model(model, item_features_local, label, test_mat=None):
    num_items = train_interactions_csr.shape[1]
    item_ids = np.arange(num_items, dtype=np.int64)
    def score_fn(user_id: int) -> np.ndarray:
        return model.predict(user_id, item_ids, item_features=item_features_local)
    metrics = evaluate_ranking(
        score_fn,
        train_interactions_csr,
        test_mat if test_mat is not None else test_interactions,
        k=K,
        use_ndcg=True,
    )
    print(
        f"{label} -> Precision@{K}: {metrics['precision_at_k']:.6f} | "
        f"Recall@{K}: {metrics['recall_at_k']:.6f} | "
        f"NDCG@{K}: {metrics['ndcg_at_k']:.6f}"
    )
    return metrics

# ====== Baseline: popularity ======
if RUN_POP_BASELINE:
    print("\n--- POPULARITY BASELINE ---")
    popularity = np.asarray(train_interactions_csr.getnnz(axis=0)).ravel().astype(float)
    def pop_score_fn(_: int) -> np.ndarray:
        return popularity
    pop_metrics = evaluate_ranking(
        pop_score_fn,
        train_interactions_csr,
        test_interactions,
        k=K,
        use_ndcg=True,
    )
    print(
        f"Popularity -> Precision@{K}: {pop_metrics['precision_at_k']:.6f} | "
        f"Recall@{K}: {pop_metrics['recall_at_k']:.6f} | "
        f"NDCG@{K}: {pop_metrics['ndcg_at_k']:.6f}"
    )

# ====== Strong-signal evaluation (rating >= 3) ======
if RUN_STRONG_EVAL:
    print("\n--- STRONG-SIGNAL EVAL (rating >= 3) ---")
    user2id_path = os.path.join(DATA_DIR_P4, "user2id.json")
    book2id_path = os.path.join(DATA_DIR_P4, "book2id.json")
    explicit_path = os.path.join("phase4_outputs", "explicit_interactions.csv")

    with open(user2id_path, "r", encoding="utf-8") as f:
        user2id = json.load(f)
    with open(book2id_path, "r", encoding="utf-8") as f:
        book2id = json.load(f)

    strong_rows = []
    strong_cols = []

    with open(explicit_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                rating = float(row.get("rating", 0))
            except ValueError:
                continue
            if rating < 3:
                continue
            uid_raw = (row.get("user_id", "") or "").strip()
            bid_raw = (row.get("book_id", "") or "").strip()
            if uid_raw not in user2id or bid_raw not in book2id:
                continue
            strong_rows.append(user2id[uid_raw])
            strong_cols.append(book2id[bid_raw])

    strong_all = sparse.coo_matrix(
        (np.ones(len(strong_rows), dtype=np.float32), (np.array(strong_rows), np.array(strong_cols))),
        shape=train_interactions_csr.shape,
    ).tocsr()

    strong_test = strong_all.multiply(test_interactions)
    print("Strong test nnz:", strong_test.nnz)
    eval_model(model, item_features, "Strong-signal", test_mat=strong_test)

# ====== Ablation: with vs without item features ======
if RUN_ABLATION:
    print("\n--- ABLATION: ITEM FEATURES ---")
    for use_features in [True, False]:
        m = LightFM(
            loss="warp",
            no_components=64,
            learning_rate=0.05,
            random_state=42,
        )
        m.fit(
            train_interactions,
            sample_weight=train_weights,
            item_features=item_features if use_features else None,
            epochs=10,
            num_threads=4,
        )
        label = "With features" if use_features else "No features"
        eval_model(m, item_features if use_features else None, label)

# ====== Loss comparison: WARP vs BPR ======
if RUN_LOSS_COMPARE:
    print("\n--- LOSS COMPARISON ---")
    for loss in ["warp", "bpr"]:
        m = LightFM(
            loss=loss,
            no_components=64,
            learning_rate=0.05,
            random_state=42,
        )
        m.fit(
            train_interactions,
            sample_weight=train_weights,
            item_features=item_features,
            epochs=10,
            num_threads=4,
        )
        eval_model(m, item_features, f"Loss={loss}")


--- SANITY CHECKS ---
Train shape: (79153, 20700)
Train nnz: 1539303
Test nnz: 389656
Item features shape: (20700, 21947)
Train/Test overlap nnz: 0


In [5]:
import itertools
import pickle
import numpy as np
from lightfm import LightFM

required_vars = ["train_interactions", "train_weights", "train_interactions_csr", "test_interactions", "item_features"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(
        "Missing data variables: " + ", ".join(missing_vars) + ". Run the training/setup cell first."
    )

# ==========================================
# EARLY-STOPPING HYPERPARAM SEARCH
# ==========================================
param_grid = {
    "no_components": [64, 128],
    "learning_rate": [0.01, 0.05],
    "loss": ["warp", "bpr"],
}

MAX_EPOCHS = 50
EVAL_EVERY = 5
PATIENCE = 3
MIN_DELTA = 1e-4
K_EVAL = 10
USE_NDCG = True

keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"🚀 Early-stopping search với {len(combinations)} cấu hình...")

best_overall = -1.0
best_params = None
results = []
num_items = train_interactions_csr.shape[1]
item_ids = np.arange(num_items, dtype=np.int64)

def eval_precision(model):
    def score_fn(user_id: int) -> np.ndarray:
        return model.predict(user_id, item_ids, item_features=item_features)
    metrics = evaluate_ranking(
        score_fn,
        train_interactions_csr,
        test_interactions,
        k=K_EVAL,
        use_ndcg=USE_NDCG,
    )
    return metrics["precision_at_k"], metrics

for i, params in enumerate(combinations):
    print(f"\n[Cấu hình {i+1}/{len(combinations)}] {params}")
    model = LightFM(
        loss=params["loss"],
        no_components=params["no_components"],
        learning_rate=params["learning_rate"],
        random_state=42,
    )
    best_score = -1.0
    best_epoch = 0
    best_state = None
    no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.fit_partial(
            train_interactions,
            sample_weight=train_weights,
            item_features=item_features,
            epochs=1,
            num_threads=4,
        )
        if epoch % EVAL_EVERY != 0:
            continue
        score, metrics = eval_precision(model)
        print(f"  Epoch {epoch}: Precision@{K_EVAL}={score:.6f}")
        if score > best_score + MIN_DELTA:
            best_score = score
            best_epoch = epoch
            best_state = pickle.dumps(model)
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            print(f"  Early stop (patience={PATIENCE}) at epoch {epoch}")
            break

    if best_state is not None:
        model = pickle.loads(best_state)
    results.append({
        "params": params,
        "best_precision": best_score,
        "best_epoch": best_epoch,
    })

    if best_score > best_overall:
        best_overall = best_score
        best_params = params
        with open("best_lightfm_model.pkl", "wb") as f:
            pickle.dump(model, f)

print("\n" + "="*50)
print("🏆 KẾT QUẢ TỐT NHẤT (Early-Stopping)")
print("="*50)
print(f"✨ Best Precision@{K_EVAL}: {best_overall:.6f}")
print(f"🛠️ Best params: {best_params}")
print("💾 Best model saved to: best_lightfm_model.pkl")

import pandas as pd
df_results = pd.DataFrame([
    {**r["params"], "best_precision": r["best_precision"], "best_epoch": r["best_epoch"]}
    for r in results
 ])
print("\nBảng so sánh chi tiết:")
print(df_results.sort_values(by="best_precision", ascending=False))

🚀 Early-stopping search với 8 cấu hình...

[Cấu hình 1/8] {'no_components': 64, 'learning_rate': 0.01, 'loss': 'warp'}


KeyboardInterrupt: 

In [ ]:
import os
import json
import numpy as np
from scipy import sparse

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN
# ==========================================
DATA_DIR_P4 = "phase4_outputs"

# Các tệp cần thiết
FEATURES_PATH = os.path.join(DATA_DIR_P4, "item_features.npz")
FEATURE_MAP_PATH = os.path.join(DATA_DIR_P4, "feature2id.json")
BOOK_MAP_PATH = os.path.join(DATA_DIR_P4, "book2id.json")
STATS_PATH = os.path.join(DATA_DIR_P4, "item_feature_stats.json")

def check_item_features():
    print("--- BẮT ĐẦU KIỂM ĐỊNH ITEM FEATURES ---\n")

    # 2. Load dữ liệu
    if not os.path.exists(FEATURES_PATH):
        print(f"❌ Không tìm thấy file: {FEATURES_PATH}")
        return

    item_features = sparse.load_npz(FEATURES_PATH)
    with open(FEATURE_MAP_PATH, "r", encoding='utf-8') as f:
        feature2id = json.load(f)
    with open(BOOK_MAP_PATH, "r", encoding='utf-8') as f:
        book2id = json.load(f)

    id2feature = {v: k for k, v in feature2id.items()}
    id2book = {v: k for k, v in book2id.items()}

    # 3. Thống kê tổng quan
    n_items, n_features = item_features.shape
    print(f"📊 Tổng số sách: {n_items}")
    print(f"📊 Tổng số thuộc tính (Features): {n_features}")

    # Tính số thuộc tính trung bình cho mỗi cuốn sách
    features_per_item = item_features.getnnz(axis=1)
    print(f"📊 Số thuộc tính trung bình mỗi cuốn sách: {np.mean(features_per_item):.2f}")
    print(f"📊 Số sách không có bất kỳ thuộc tính nào: {np.sum(features_per_item == 0)}")

    # 4. Phân tích loại thuộc tính (Feature Types)
    # Giả sử định dạng là "loại:giá_trị" (vd: "author:J.K.Rowling")
    feature_types = {}
    for feat_name in feature2id.keys():
        if ":" in feat_name:
            prefix = feat_name.split(":")[0]
            feature_types[prefix] = feature_types.get(prefix, 0) + 1
        else:
            feature_types["other/unlabeled"] = feature_types.get("other/unlabeled", 0) + 1

    print("\n--- PHÂN LOẠI THUỘC TÍNH ĐANG CÓ ---")
    if not feature_types:
        print("⚠️ Cảnh báo: Không tìm thấy tiền tố (loại:). Có thể bạn chưa gắn nhãn thuộc tính.")
    for ftype, count in feature_types.items():
        print(f"🔹 {ftype}: {count} giá trị khác nhau")

    # 5. Kiểm tra ngẫu nhiên 3 cuốn sách
    print("\n--- KIỂM TRA NGẪU NHIÊN 3 CUỐN SÁCH ---")
    sample_indices = np.random.choice(n_items, 3, replace=False)

    for idx in sample_indices:
        book_id = id2book.get(idx, "Unknown")
        row = item_features[idx]
        f_indices = row.indices
        f_names = [id2feature[i] for i in f_indices]

        print(f"\n📖 Sách Index {idx} (ID: {book_id}):")
        if not f_names:
            print("   ⚠️ Trống (Không có thuộc tính)")
        else:
            for name in f_names:
                print(f"   - {name}")

    # 6. Đọc tệp Stats nếu có
    if os.path.exists(STATS_PATH):
        print("\n--- TÓM TẮT TỪ FILE STATS ---")
        with open(STATS_PATH, "r") as f:
            stats = json.load(f)
            print(json.dumps(stats, indent=2))

if __name__ == "__main__":
    check_item_features()
